# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters and compare the NN performance to a simple logistic regression (a single neuron). 

For other comparisons, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types)
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


Note for solution: below the batch approach is used - non-batch process are the cells commented out.

0) Import dependencies and datasets

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)

True
NVIDIA RTX 1000 Ada Generation Laptop GPU
cuda


1) Load and investigate the data

In [6]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [7]:
def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def rdkitfp(mol):
    fp = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def atompairfp(mol):
    fp = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

fpgens = {
    "MorganFP": morganfp,
    "RDKitFP": rdkitfp,
    "AtomPairFP": atompairfp,
    "MACCSkeys": maccskeys
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,RDKitFP,AtomPairFP,MACCSkeys
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x0000018A353...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x0000018A353...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x0000018A353...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x0000018A353...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x0000018A353...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [ ]:
# Alternative way provided - see below
# X = np.stack(df["MorganFP"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
# y = df["mutagenicity"].to_numpy()

# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.float32)
# X_test = torch.tensor(X_test, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.float32)

In [36]:
X = df["MorganFP"]
y = df["mutagenicity"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataFromFPinDF(Dataset):
    def __init__(self, X, y):
        # features provided as in the Dataframe, fingerprints as single arrays in each cell
        if len(X.shape) == 1:
            self.X = torch.tensor(
                np.stack(df["MorganFP"].values)
                # dtype=torch.float32() # optionally include a datatype
                )
        elif len(X.shape) == 2:
            self.X = torch.tensor(    
                X.to_numpy(), 
                # dtype=torch.float32() # optionally include a datatype
                ) 
        
        # targets provided as Dataframe
        if len(y.shape) == 1:
            self.y = torch.tensor(
                np.stack(df["mutagenicity"].values),
                dtype=torch.float32 # optionally include a datatype
                )
        elif len(y.shape) == 2:
            self.y = torch.tensor(    
                y.to_numpy(), 
                dtype=torch.float32 # optionally include a datatype
                ) 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)
    

train_dataset = MyDataFromFPinDF(X_train, y_train)
test_dataset  = MyDataFromFPinDF(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [103]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()

        # define your model width and depth below
        self.fc1 = nn.Linear(2048, 256)
        self.dropout1 = nn.Dropout(p=0.3)

        self.fc2 = nn.Linear(256, 64)
        self.dropout2 = nn.Dropout(p=0.3)

        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)

        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)

        x = self.fc3(x)
        return x


In [104]:
# Hyperparameters
# input_size = X_train.shape[1]
# hidden_size = 20
# output_size = 1
learning_rate = 0.01

# Number of training iterations
num_epochs = 100

In [ ]:
import torch.optim as optim
model = BinClassifierNN()
# model = BinClassifierNN().to(device)

criterion = nn.BCEWithLogitsLoss()  # loss function defined for binary classification

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [106]:
# alternative approach for loading directly the training tensors
# for epoch in range(num_epochs):
   
#     model.train()
#     outputs = model(X_train).squeeze()
#     loss = criterion(outputs, y_train)

#     optimizer.zero_grad() # clear existing gradients
#     loss.backward() # backpropagation of loss function to optimise gradient
#     optimizer.step() # update parameters using Adam

#     if (epoch + 1) % 10 == 0:
#         print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

In [ ]:
# batched training
for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    total_samples = 0

    for data, targets in train_loader:
        # data = data.to(device)
        # targets = targets.to(device)

        outputs = model(data.to(torch.float32)).squeeze()

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # accumulate properly
        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    avg_loss = total_loss / total_samples

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}')

Epoch [10/100], Loss: 0.0166
Epoch [20/100], Loss: 0.0056
Epoch [30/100], Loss: 0.0081
Epoch [40/100], Loss: 0.0271
Epoch [50/100], Loss: 0.0055
Epoch [60/100], Loss: 0.0170
Epoch [70/100], Loss: 0.0167
Epoch [80/100], Loss: 0.0091
Epoch [90/100], Loss: 0.0213
Epoch [100/100], Loss: 0.0098


6) Evaluate the model. As a first metric, you can use the same Loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [ ]:
# Evaluate the model - alternative process with batch-size provided below
# bce_loss = nn.BCEWithLogitsLoss()

# model.eval()

# with torch.no_grad():
#     y_pred = model(X_test).squeeze()
#     bce = bce_loss(y_pred, y_test).item()
#     print("BCE_Loss:", bce)

In [ ]:
# Evaluate the model
model.eval()  # set model to eval mode (e.g. no dropout)

all_preds = []
all_targets = []

total_samples = 0
total_loss = 0

criterion = nn.BCEWithLogitsLoss()

with torch.no_grad():
    for data, targets in test_loader:
        # data = data.to(device)
        # targets = targets.to(device)

        outputs = model(data.to(torch.float32)).squeeze()
        loss = criterion(outputs, targets)

        total_loss += loss.item() * data.size(0) # multiply by batch size to 
        total_samples += targets.size(0)
        
        # Predictions      
        probs = torch.sigmoid(outputs)     # convert for metrics
        preds = (probs > 0.5).long()

        all_preds.extend(preds.cpu().tolist()) # cpu() moves the tensor to cpu (if on gpu), tolist converts it to python list
        all_targets.extend(targets.cpu().tolist())

avg_loss = total_loss / total_samples

print(f"Average test Loss: {avg_loss:.4f}")


Average test Loss: 0.0016


In [ ]:
# Evaluate the model with scikit-metrics
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

print("Accuracy:", accuracy_score(all_targets, all_preds))
print("F1:", f1_score(all_targets, all_preds))
print("ROC-AUC:", roc_auc_score(all_targets, all_preds))
print("Precision:", precision_score(all_targets, all_preds))
print("Recall:", recall_score(all_targets, all_preds))
print("Confusion:\n", confusion_matrix(all_targets, all_preds))

Accuracy: 0.9989007969222314
F1: 0.998992443324937
ROC-AUC: 0.9989934574735783
Precision: 1.0
Recall: 0.9979869149471565
Confusion:
 [[3304    0]
 [   8 3966]]


In [ ]:
# # Evaluate the model
# from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

# model.eval()
# with torch.no_grad():
#     outputs = model(X_test).squeeze()
#     y_probs = torch.sigmoid(outputs)     # convert for metrics
#     y_preds = (y_probs > 0.5).long()
#     # y_pred = (y_probs > 0.5).int()

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("F1:", f1_score(y_test, y_pred))
# print("ROC-AUC:", roc_auc_score(y_test, y_probs))
# print("Precision:", precision_score(y_test, y_pred))
# print("Recall:", recall_score(y_test, y_pred))
# print("Confusion:\n", confusion_matrix(y_test, y_pred))

TypeError: linear(): argument 'input' (position 1) must be Tensor, not Series

7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [190]:
# saving a checkpoint

checkpoint = {
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch,
    "loss": loss
}

torch.save(checkpoint, "model_checkpoint.pth")


In [191]:

# load the checkpoint again

checkpoint = torch.load("model_checkpoint.pth")

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

epoch = checkpoint["epoch"]
loss = checkpoint["loss"]


In [192]:
# Save the entire model
torch.save(model, "model_full.pth")

In [194]:
# Load a full model
model = torch.load("model_full.pth", weights_only=False)
model.eval()

BinClassifierNN(
  (fc1): Linear(in_features=167, out_features=256, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=64, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)

#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?